## Install needed libraries

In [1]:
# !pip install emoji
# !pip install --upgrade bitsandbytes
# !pip install evaluate

## Imports

In [2]:
import pandas as pd
import emoji
import re
import torch
from itertools import chain
from datasets import Dataset, Features, Value
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    GenerationConfig
)
import bitsandbytes as bnb
import evaluate

## Connect google drive to load data

In [3]:
# from google.colab import drive

# drive.mount("/content/drive")

## Load data

In [4]:
# path to folder with all data
DATA_PATH = "./drive/MyDrive/hw_hf_transformers"

I have two kinda parts of data: with us_gb prefix (data from kaggle: https://www.kaggle.com/tanmay111/youtube-comments-sentiment-analysis/data?select=UScomments.csv) and with cstm_prefix (i collected this data on my own).

Each part is also divided into 2 other parts: with emojis (in video title or in comment or in channel title) and without them. I will finetune gpt2-large on some part of the data with emojis (because it especially common for comments to use them) and on some part of the data without them.

In [5]:
df_us_gb_with_emoji = pd.read_csv(f"{DATA_PATH}/final_grouped_dataset_us_gb_with_emoji.csv")
df_cstm_with_emoji = pd.read_csv(f"{DATA_PATH}/final_grouped_dataset_cstm_with_emoji.csv")

df_us_gb_without_emoji = pd.read_csv(f"{DATA_PATH}/final_grouped_dataset_us_gb_without_emoji.csv")
df_cstm_without_emoji = pd.read_csv(f"{DATA_PATH}/final_grouped_dataset_cstm_without_emoji.csv")

In [6]:
print(f"Number of observations (comments) in data from kaggle without emoji: {df_us_gb_without_emoji.shape[0]}")
print(f"Number of observations (comments) in data from kaggle with emoji: {df_us_gb_with_emoji.shape[0]}")
print(f"Number of observations (comments) in data collected by me without emoji: {df_cstm_without_emoji.shape[0]}")
print(f"Number of observations (comments) in data from kaggle with emoji: {df_cstm_with_emoji.shape[0]}")

Number of observations (comments) in data from kaggle without emoji: 448574
Number of observations (comments) in data from kaggle with emoji: 70375
Number of observations (comments) in data collected by me without emoji: 604361
Number of observations (comments) in data from kaggle with emoji: 227218


Let's look on data with emoji in

In [7]:
df_cstm_with_emoji.head(10)

,Unnamed: 0,title,channel_title,comment_text
0,0,The cutest animal rescue you'll ever see 😍,We Love Animals,Such a little HERO she deserves an award for s...
1,1,The cutest animal rescue you'll ever see 😍,We Love Animals,A HUGE Hero in a pretty little princess 😊 👏 👏 👏
2,3,The cutest animal rescue you'll ever see 😍,We Love Animals,Yikes. Someone should of helped her. Too much ...
3,4,The cutest animal rescue you'll ever see 😍,We Love Animals,"look at the innocent face, so cute"
4,5,Good People Save a Tiny Life ❤️‍🩹🦔,We Love Animals,"No AI voices, it kills the mood"
5,6,Good People Save a Tiny Life ❤️‍🩹🦔,We Love Animals,Guys just change the audio track then the ai v...
6,7,Good People Save a Tiny Life ❤️‍🩹🦔,We Love Animals,I would have jumped in just to save a life
7,8,Good People Save a Tiny Life ❤️‍🩹🦔,We Love Animals,Just happened to have that special broom sitti...
8,9,Good People Save a Tiny Life ❤️‍🩹🦔,We Love Animals,What a beautiful thing to be a part of. God bless
9,10,Good People Save a Tiny Life ❤️‍🩹🦔,We Love Animals,Kind hearts All Around The World! 🌎❣️


Let's look on data without emoji

In [8]:
df_cstm_without_emoji.head(10)

,Unnamed: 0,title,channel_title,comment_text
0,59,Missing dog finally found after 2 years at Lou...,We Love Animals,The poor dog probably thought she abandoned her.
1,60,Missing dog finally found after 2 years at Lou...,We Love Animals,That's always so beautiful when animals are re...
2,61,Missing dog finally found after 2 years at Lou...,We Love Animals,I am glad they were reunited!! Thank you for ...
3,63,Missing dog finally found after 2 years at Lou...,We Love Animals,My God I would be a emotional wreck if I lost ...
4,65,Missing dog finally found after 2 years at Lou...,We Love Animals,So glad they were reunited!
5,66,Missing dog finally found after 2 years at Lou...,We Love Animals,She never gave up hope that her owner would on...
6,71,Missing dog finally found after 2 years at Lou...,We Love Animals,That is great to see
7,76,Missing dog finally found after 2 years at Lou...,We Love Animals,Thanks \nBlessings
8,78,Missing dog finally found after 2 years at Lou...,We Love Animals,I would of been cryyyyyyyyy'N \nOmg 2 yrs really!
9,81,Missing dog finally found after 2 years at Lou...,We Love Animals,Imagine there was a chip that contains all the...


## Format all texts for our model

In [9]:
def format_example(row):
    return f"<CHANNEL> {row["channel_title"]} <TITLE> {row["title"]} <COMMENT> {row["comment_text"]}"

df_us_gb_with_emoji["formatted"] = df_us_gb_with_emoji.apply(lambda row: format_example(row), axis=1)
df_us_gb_without_emoji["formatted"] = df_us_gb_without_emoji.apply(lambda row: format_example(row), axis=1)
df_cstm_with_emoji["formatted"] = df_cstm_with_emoji.apply(lambda row: format_example(row), axis=1)
df_cstm_without_emoji["formatted"] = df_cstm_without_emoji.apply(lambda row: format_example(row), axis=1)

Define some parameters and also model and tokenizator

In [10]:
# Column with final formatted text in huggingface datasets
text_column_name = "text"

# Name of the model
model_name_or_path = "openai-community/gpt2-large"

# Define tokenizator
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

# Define model
model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    use_cache=False,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2-large
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
tokenizer.bos_token == tokenizer.eos_token

True

# Create datasets

In [12]:
# Create datasets
texts_us_gb_without_emoji = (df_us_gb_without_emoji["formatted"] + tokenizer.eos_token).tolist()
texts_cstm_without_emoji = (df_cstm_without_emoji["formatted"] + tokenizer.eos_token).tolist()

texts_us_gb_with_emoji = (df_us_gb_with_emoji["formatted"] + tokenizer.eos_token).tolist()
texts_cstm_with_emoji = (df_cstm_with_emoji["formatted"] + tokenizer.eos_token).tolist()

# For the finetuning we use some amount data with emojis and a some amount of data without them
data_to_use = texts_us_gb_with_emoji[:30000] + texts_cstm_with_emoji + texts_us_gb_without_emoji[:100000] + texts_cstm_without_emoji[:100000]
real_dataset = Dataset.from_dict({"text": data_to_use})

In [13]:
print(f"Length of final_dataset (not yet divided into train/val) = {len(real_dataset)}")

Length of final_dataset (not yet divided into train/val) = 457218


## Add special tokens to tokenizator, change pad_side, resize_model_embeddings

In [14]:
special_tokens = {
    "additional_special_tokens": ["<CHANNEL>", "<TITLE>", "<COMMENT>"],
    "pad_token": "<PAD>"
}

# Add special tokens
tokenizer.add_special_tokens(special_tokens)

# Padding side would be from left
tokenizer.padding_side = "left"

model.config.pad_token_id = tokenizer.pad_token_id

# Extend the model embeddings for new tokens
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50261, 1280)

In [15]:
# tokenizer.eos_token_id

## Define function for tokenization

In [16]:
def tokenize_function(example):
    tokens = tokenizer(
        example[text_column_name],
        truncation=True,
        max_length=256,
    )

    # labels left for collator

    return tokens

Get final tokenized dataset

In [17]:
final_dataset = real_dataset.shuffle(seed=42).map(tokenize_function, batched=True, batch_size=100).remove_columns("text")

Map:   0%|          | 0/457218 [00:00<?, ? examples/s]

In [18]:
final_dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 457218
})

## Split into train and test

In [19]:
split_dataset = final_dataset.train_test_split(test_size=0.05, shuffle=True, seed=42)

print(f"Train: {len(split_dataset["train"])}")
print(f"Val: {len(split_dataset["test"])}")

Train: 434357
Val: 22861


# Define Training arguments

In [20]:
training_args = TrainingArguments(
    output_dir="./youtube-gpt2-fullfinetuned",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    eval_accumulation_steps=4,
    num_train_epochs=3,
    max_steps=-1, #
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps=1000, # calculate from total number of steps
    weight_decay=0.01,
    logging_steps=1000,
    save_steps=1000,
    eval_strategy="steps",
    eval_steps=1000, # 1000
    save_total_limit=2,
    report_to="none",
    disable_tqdm=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    seed=42,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    bf16=True,
)

# define optimizer
optimizer = bnb.optim.Adam8bit(model.parameters(), lr=3e-5)

## Let's fine tune the model

In [21]:
trainer = Trainer(
    model,
    training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    optimizers=(optimizer, None),
)

fine tune

In [22]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
1000,3.746500,2.333356
2000,2.239426,2.068951
3000,2.082373,1.991497
4000,2.018659,1.956639
5000,1.987298,1.929433
6000,1.959798,1.906957
7000,1.920087,1.898343
8000,1.827417,1.887344
9000,1.821350,1.881618
10000,1.817730,1.876431


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=20361, training_loss=1.9325057558481518, metrics={'train_runtime': 7550.132, 'train_samples_per_second': 172.589, 'train_steps_per_second': 2.697, 'total_flos': 4.283759569527552e+17, 'train_loss': 1.9325057558481518, 'epoch': 3.0})

Let's look on results: evaluation loss and perplexity

In [52]:
evaluation_results = trainer.evaluate()

print(f"Loss after fine-tuning on validation data is {evaluation_results["eval_loss"]}")
print(f"Perplexity after fine-tuning on validation data is {torch.exp(torch.tensor(evaluation_results["eval_loss"])).item()}")

Loss after fine-tuning on validation data is 1.8545504808425903
Perplexity after fine-tuning on validation data is 6.3888258934021


We define example config for generation. And generate some sequences

In [59]:
generation_config = GenerationConfig(
    max_new_tokens=128,
    min_new_tokens=2,
    temperature=0.7,
    do_sample=True,
    # num_beams=6,
    top_p=0.9,
    num_return_sequences=3,
    eos_token_id=tokenizer.eos_token_id,
    # pad_token_id=tokenizer.pad_token_id,
    repetition_penalty=1.5,
)

In [60]:
prompt = "<CHANNEL> BBC <TITLE> Trump attacked GERMANY! <COMMENT>"
inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    inputs=inputs["input_ids"].to(model.device),
    attention_mask=inputs["attention_mask"].to(model.device),
    generation_config=generation_config,
)

for i, seq in enumerate(output):
    text = tokenizer.decode(seq, skip_special_tokens=False)
    comment = text.split("<COMMENT>")[-1].replace(tokenizer.eos_token, "").strip()
    print(f"{i+1}: {comment} [THE END]\n")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


1: I'm sure that all of you, who are German citizens will never forget what happened to the United States during and after World War II. You Americans were not only defeated by Hitler but also humiliated in a series of humiliating defeats at the hands of our allies. We Germans have suffered greatly under your rule over many years... [THE END]

2: The United States of America is not a democracy. It's an authoritarian state, and that means it has to go down the drain because there are no checks on its power or actions against citizens' rights (in any form). [THE END]

3: The US and Israel are now actively seeking a "buffer zone" between Iran (which they already control) and the rest of Middle East. This is more than an empty threat; it's about blackmailing Europe, Russia, China & North Korea into submission with oil prices as their leverage... [THE END]



In [23]:
# model.save_pretrained("./full_finetuned_best")
# tokenizer.save_pretrained("./full_finetuned_best")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./full_finetuned_best/tokenizer_config.json',
 './full_finetuned_best/tokenizer.json')